In [20]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [21]:
eos_token = "<|endoftext|>"

def clean_and_filter(example):
    if len(example['text'].strip()) < 20:
        return False
    example['text'] = example['text'].strip() + eos_token
    return True

dataset = dataset.filter(clean_and_filter)

In [22]:
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling

Загрузка файла: data/TinyStoriesV2-GPT4-valid.txt
Всего токенов в датасете: 5,532,630
Создано семплов: 29651
Размер датасета: 29651
Информация о токенайзере: {'tokenizer_name': 'p50k_base', 'vocab_size': 50281, 'eot_token_id': 50256, 'pad_token_id': 50256, 'max_length': 256}
Input shape: torch.Size([256])
Labels shape: torch.Size([256])
Первые 10 input токенов: [84, 836, 470, 423, 284, 307, 12008, 286, 262, 7812]
Первые 10 label токенов: [836, 470, 423, 284, 307, 12008, 286, 262, 7812, 3290]


In [24]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_seq_len = 512

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_len,
        return_attention_mask=True,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Batch input shape: tensor([[  198,  7454,  2402,   257,   640,    11,   612,   373,   257,  1310,
          3290,  3706,  5436,    13,  5436,   373,   845,  6507,   290, 22444,
           780,   339,   550,   645,  2460,   284,   711,   351,    13,   679,
           561,  2513,  1088,   262,  3952,   477,  3436,    11,  2045,   329,
          2130,   284, 16225,   290,   423,  1257,   351,    13,   198,  3198,
          1110,    11,  5436,  2497,   257,  1310,  2576,  3706, 18966,    13,
         18966,   373,   635,  6507,   290, 22444,    11,  5586,   319,   257,
          7624,   477,   416,  5223,    13,  5436,  1807,   326,  3863,   611,
           339,  2921,   607,   257, 16225,    11,   673,   561,   307,   465,
          1545,    13,  1406,    11,   339,  6807,   510,   284,   607,   290,
         41130,   607,  1232,    13,   198, 10161,  2611,  2936,   262, 16225,
           290, 13541,    13,  1375, 41130,  5436,   736,    11,   290,   484,
          1111,  2936,  3772,    

In [25]:
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    examples["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in seq]
        for seq in examples["labels"]
    ]
    return examples

tokenized_dataset = tokenized_dataset.map(add_labels, batched=True)

In [ ]:
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Теперь DataLoader работает без collate_fn:
from torch.utils.data import DataLoader

train_data = list(tokenized_dataset["train"])   # весь датасет в ОЗУ
train_loader = DataLoader(
    train_data,
    batch_size=4,
    shuffle=True,
    pin_memory=True
    #num_workers=2
)

In [ ]:
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

In [ ]:
vocab_size = 50257          # как у gpt2
n_positions = 512           # max_seq_len (будем использовать 256)
n_embd = 768                # размерность эмбеддингов
n_layer = 12                 # количество слоёв Transformer
n_head = 8                  # количество голов внимания

config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=n_positions,
    n_embd=n_embd,
    n_layer=n_layer,
    n_head=n_head,
)
model = GPT2LMHeadModel(config)
print(f"Параметров: {model.num_parameters():,}")

In [ ]:
import torch

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

num_epochs = 10
model.train()

In [ ]:
from tqdm import tqdm

In [ ]:
for epoch in range(num_epochs):
    total_loss = 0
    for step, batch in tqdm(enumerate(train_loader)):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if step % 100 == 0:
            print(f"Epoch {epoch}, step {step}, loss = {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Average train loss: {avg_train_loss:.4f}")